# Chain-of-Thought (CoT) Study - 5. BT Russian (ru)

This notebook runs Back-Translated QA for **Russian (ru)** using **P2-cot** strategy.

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'torch', 'accelerate'], check=True)

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
else:
    PROJECT_ROOT = os.getcwd()

if not os.path.exists(PROJECT_ROOT) and (IN_KAGGLE or IN_COLAB):
    subprocess.run(['git', 'clone', 'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git', PROJECT_ROOT], check=True)

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('✓ Qwen cached')

In [ ]:
ABLATION_DIR = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/prompt-ablation"
COT_DIR = f"{ABLATION_DIR}/cot"
CODE_DIR = f"{ABLATION_DIR}/code"

STRATEGY = "P2-cot"
LANG = "ru"
BT_PATH = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/baseline/BT/{LANG}.jsonl"

os.makedirs(f"{COT_DIR}/QA", exist_ok=True)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

In [ ]:
output_file = f"{COT_DIR}/QA/{LANG}-{STRATEGY}.jsonl"

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/qa_ablation.py",
    "--strategy", STRATEGY,
    "--mode", "bt",
    "--bt_input_path", BT_PATH,
    "--lang", LANG,
    "--output_path", output_file
]

print(f"Running BT QA for {LANG} with {STRATEGY}...")
subprocess.run(cmd, check=True)
print(f"✓ BT QA {LANG} complete!")

In [ ]:
os.chdir(PROJECT_ROOT)
subprocess.run(['git', 'add', '-A'])
subprocess.run(['git', 'commit', '-m', f'Add CoT BT QA results for {LANG}'])
subprocess.run(['git', 'push', 'origin', 'main'])
print("✓ Push complete!")